In [2]:
import h5py
import pandas as pd
import glob
import numpy as np
import json

In [3]:
#result_paths = ['/data/wiay/2297403c/amaze_model_select/Nflows_AMAZE_paper/outputs/cont_GWTC3/prod_retrainedCE',\
#    '/data/wiay/2297403c/amaze_model_select/Nflows_AMAZE_paper/outputs/discrete_GWTC3/flow_retrainedCE',
#    '/data/wiay/2297403c/amaze_model_select/Nflows_AMAZE_paper/outputs/discrete_GWTC3/KDEs']

result_paths = ['/data/wiay/2297403c/amaze_model_select/Nflows_AMAZE_paper/outputs/cont_GWTC3/prod_longsampling']
removing_keys = ['model_selection/detectable_samples']

for path in result_paths:
    results = glob.glob(path+'/*.hdf5')

    for result in results:
        print(result)
        with h5py.File(result,  "a") as f:
            try:
                del f[removing_keys[0]]
            except:
                continue
            print(result)
            print(f['model_selection'].keys())
            f.close()

/data/wiay/2297403c/amaze_model_select/Nflows_AMAZE_paper/outputs/cont_GWTC3/prod_longsampling/output_seed57.hdf5
/data/wiay/2297403c/amaze_model_select/Nflows_AMAZE_paper/outputs/cont_GWTC3/prod_longsampling/output_seed29.hdf5
/data/wiay/2297403c/amaze_model_select/Nflows_AMAZE_paper/outputs/cont_GWTC3/prod_longsampling/output_seed14.hdf5
/data/wiay/2297403c/amaze_model_select/Nflows_AMAZE_paper/outputs/cont_GWTC3/prod_longsampling/output_seed96.hdf5
/data/wiay/2297403c/amaze_model_select/Nflows_AMAZE_paper/outputs/cont_GWTC3/prod_longsampling/output_seed28.hdf5
/data/wiay/2297403c/amaze_model_select/Nflows_AMAZE_paper/outputs/cont_GWTC3/prod_longsampling/output_seed40.hdf5
/data/wiay/2297403c/amaze_model_select/Nflows_AMAZE_paper/outputs/cont_GWTC3/prod_longsampling/output_seed13.hdf5
/data/wiay/2297403c/amaze_model_select/Nflows_AMAZE_paper/outputs/cont_GWTC3/prod_longsampling/output_seed80.hdf5
/data/wiay/2297403c/amaze_model_select/Nflows_AMAZE_paper/outputs/cont_GWTC3/prod_longsa

In [32]:
#error with /data/wiay/2297403c/amaze_model_select/Nflows_AMAZE_paper/outputs/discrete_GWTC3/KDEs/output_seed39.hdf5
with h5py.File('/data/wiay/2297403c/amaze_model_select/output_seed39.hdf5',  "a") as f:
    del f[removing_keys[0]]
    print(result)
    print(f['model_selection'].keys())

/data/wiay/2297403c/amaze_model_select/Nflows_AMAZE_paper/outputs/discrete_GWTC3/KDEs/output_seed39.hdf5
<KeysViewHDF5 ['lnprb', 'obsdata', 'raw_samples', 'samples']>


In [4]:
#writing flow mappings as JSON file
channels = ['CE', 'CHE', 'GC', 'NSC', 'SMT']

for channel in channels:
    mappings = np.load(f'/data/wiay/2297403c/amaze_model_select/Nflows_AMAZE_paper/inputs/flow_models/mixed_models/{channel}_mappings.npy')
    print(mappings)
    channel_config = {'Mmax':mappings[0], 'Mup':mappings[1],'qmax':mappings[2],'qup':mappings[3], 'zmax':mappings[4], 'zup':mappings[5]}
    channel_json = {}
    channel_json[channel] = channel_config

    #write this channels config to file
    '''with open(f'/data/wiay/2297403c/amaze_model_select/Nflows_AMAZE_paper/inputs/flow_models/mixed_models/flow_mappings.json', 'a') as f:
        json.dump(channel_json, f)'''

[ -0.50543649 100.          13.34709027   1.          14.03557119
  10.        ]
[ -0.48140259 100.          14.01969998   1.           8.6194462
  10.        ]
[  0.83147742 100.           6.90775528   1.001        4.22707981
  10.        ]
[  2.37249913 100.           9.04990962   1.           0.63633117
  10.        ]
[ -0.50663736 100.          11.73507779   1.           7.09703192
  10.        ]


In [13]:
#combined detectable beta hdf5s
columns = ['p0','p1']
channels = ['CE', 'CHE', 'GC', 'NSC', 'SMT']

for channel in channels:
    columns.append('beta_'+channel)

outdir = '/data/wiay/2297403c/amaze_model_select/Nflows_AMAZE_paper/plots/prod_091224'
extra_result = h5py.File('/data/wiay/2297403c/amaze_model_select/Nflows_AMAZE_paper/plots/prod_091224/data/continuous_extra_detectable_betas.hdf5')
extra_samps = extra_result['model_selection']['detectable_samples']['block0_values']
first_result = h5py.File('/data/wiay/2297403c/amaze_model_select/Nflows_AMAZE_paper/plots/prod_091224/data/cont_retrainedCE_detectable_betas.hdf5', 'a')
first_samps = first_result["model_selection"]["detectable_samples"]['block0_values']

df = pd.DataFrame(np.vstack([first_samps,extra_samps]), columns=columns)
df.to_hdf(f'{outdir}/data/cont_combined_detectable_betas.hdf5', key='model_selection/detectable_samples')

In [4]:
#writing dataspace samps as hdf5 file
channels = ['CE', 'CHE', 'GC', 'NSC', 'SMT']
params = ['mchirp','q','chieff','z']

outdir='/data/wiay/2297403c/amaze_model_select/Nflows_AMAZE_paper/plots/prod_091224/'
total_samps_ordered = np.load(f"{outdir}/data/flow_samps_dataspace.npy")
samps_filled = np.load(f'{outdir}/data/no_channelsamps_dataspace.npy')

with h5py.File(f'{outdir}/data/dataspace_samps_extrafiles.hdf5', 'w') as f:
    f.create_group("flow_samps")
    f.create_group("parametric_samps")

for cidx, channel in enumerate(channels):
    df = pd.DataFrame(total_samps_ordered[samps_filled[cidx-1]:samps_filled[cidx]], columns=params)
    df.to_hdf(f'{outdir}/data/dataspace_samps_extrafiles.hdf5', key=f'flow_samps/{channel}')

comb_intrins_samps = pd.read_hdf('/data/wiay/2297403c/GW_ChirpSim/binary_param_generation/60_intrins_samps_zmax1_35.hdf5', key='all_intrins_samps')
mchirp_samps = comb_intrins_samps['m1']*(comb_intrins_samps['q']**3/(1+comb_intrins_samps['q']))
chieff_samps = ((comb_intrins_samps['chi_1']*comb_intrins_samps['costilt_1'])+(comb_intrins_samps['q']*comb_intrins_samps['chi_2']*comb_intrins_samps['costilt_2']))\
/(1+comb_intrins_samps['q'])

PLPP_samps = [mchirp_samps, comb_intrins_samps['q'], chieff_samps, comb_intrins_samps['z']]
df = pd.DataFrame(np.array(PLPP_samps).T, columns=params)
df.to_hdf(f'{outdir}/data/dataspace_samps_extrafiles.hdf5', key=f'parametric_samps')
        

In [52]:
total_samps_ordered[samps_filled[cidx-1]:samps_filled[cidx]].shape

(62490, 4)

In [5]:
#writing corner flow/KDE samps as hdf5 file
channels = ['CE']
params = ['mchirp','q','chieff','z']

outdir='/data/wiay/2297403c/amaze_model_select/Nflows_AMAZE_paper/plots/prod_091224'
corner_samps_flow = np.load(f"{outdir}/data/CE_flows_cornersample.npy")
corner_samps_flow_test = np.load(f"{outdir}/data/CE_flows_cornersample_testCE.npy")
corner_samps_kde = np.load(f'{outdir}/data/CE_KDEs_cornersample.npy')

with h5py.File(f'{outdir}/data/emulation_samps.hdf5', 'w') as f:
    f.create_group("flow_samps")
    f.create_group("KDE_samps")

df = pd.DataFrame(corner_samps_flow, columns=params)
df.to_hdf(f'{outdir}/data/emulation_samps.hdf5', key=f'flow_samps')
df = pd.DataFrame(corner_samps_kde, columns=params)
df.to_hdf(f'{outdir}/data/emulation_samps.hdf5', key=f'KDE_samps')


with h5py.File(f'{outdir}/data/test_flow_samps.hdf5', 'w') as f:
    f.create_group("flow_samps")

df = pd.DataFrame(corner_samps_flow_test, columns=params)
df.to_hdf(f'{outdir}/data/emulation_samps.hdf5', key=f'flow_samps')


In [4]:
#write KLs/KS json
channels = ['CE', 'CHE', 'GC', 'NSC', 'SMT']
chi_b = [0.,0.1,0.2,0.5]
alpha_CE = [0.2,.5,1.,2.,5.]
submodels_dict = {0: {0: 'chi00', 1: 'chi01', 2: 'chi02', 3: 'chi05'}, 1: {0: 'alpha02', 1: 'alpha05', 2: 'alpha10', 3: 'alpha20', 4: 'alpha50'}}

for channel in ['CE', 'CHE', 'GC', 'NSC', 'SMT']:

    flow_KL= np.load(f'/data/wiay/2297403c/amaze_model_select/Nflows_AMAZE_paper/plots/prod_091224/data/{channel}_flow_KL.npy')
    kde_KL= np.load(f'/data/wiay/2297403c/amaze_model_select/Nflows_AMAZE_paper/plots/prod_091224/data/{channel}_KDE_KL.npy')

    if channel == 'CE':
        flow_KL_dict = dict((f'({chi_b[j]},{alpha_CE[i]})', flow_KL[j][i]) for i in range(len(alpha_CE)) for j in range(len(chi_b)))
        kde_KL_dict = dict(((f'({chi_b[j]},{alpha_CE[i]})', kde_KL[j][i]) for i in range(len(alpha_CE)) for j in range(len(chi_b))))
        flow_KS= np.load(f'/data/wiay/2297403c/amaze_model_select/Nflows_AMAZE_paper/plots/prod_091224/data/{channel}_flow_KS.npy')
        kde_KS= np.load(f'/data/wiay/2297403c/amaze_model_select/Nflows_AMAZE_paper/plots/prod_091224/data/{channel}_KDE_KS.npy')
        flow_KS_dict = dict((f'({chi_b[j]},{alpha_CE[i]})', flow_KS[j][i]) for i in range(len(alpha_CE)) for j in range(len(chi_b)))
        kde_KS_dict = dict(((f'({chi_b[j]},{alpha_CE[i]})', kde_KS[j][i]) for i in range(len(alpha_CE)) for j in range(len(chi_b))))
        
    else:
        flow_KL_dict = dict(zip(chi_b, flow_KL.T))
        kde_KL_dict = dict(zip(chi_b, kde_KL.T))
    
    channel_json = {}
    if channel == 'CE':
        channel_json[channel] = {'flow KL':flow_KL_dict, 'KDE KL':kde_KL_dict, 'flow KS':flow_KS_dict, 'KDE KS':kde_KS_dict}
    else:
        channel_json[channel] = {'flow KL':flow_KL_dict, 'KDE KL':kde_KL_dict}
        
    with open(f'KLs_Kss.json', 'a') as f:
        json.dump(channel_json, f)
